# Backward pass Postmartem analysis

- how to quantify how wrong we were in foarward pass 
- We compare our guess to the truth then we figure out exactly who to blame

## 📻 Radio Tuning Analogy

Think of training a model like **tuning a radio**.

- **`W` (weight)** → Main tuning knob
- **`b` (bias)** → Fine-adjustment knob
- **`y_hat`** → Signal currently produced by the radio
- **`y_true`** → Clear signal we want
- **Loss** → How much the current signal is off
- **Training** → Repeatedly adjusting `W` and `b`

The backward pass is figuring out which direction to turn each knob to make static go away
 
The model starts with random settings:

```text
W = random
b = random

## Loss

We need a **single number** that tells us, overall, how badly our model is performing.

This number is called the **loss**.

### Simple idea

```text
Prediction (y_hat)  ──┐
                      ├──→ Loss
Actual value (y)   ───┘

## Mean Squared Error (MSE)

For **regression**, a common loss function is **Mean Squared Error (MSE)**.

MSE measures how far our predictions are from the actual values.

$$
MSE = \frac{1}{N}\sum_{i=1}^{N}(y_i - \hat{y}_i)^2
$$

### Simple idea

1. Find the difference between **actual** and **prediction**.
2. Square the difference.
3. Do this for all data points.
4. Take the average.

```text
Actual → 10
Prediction → 8

Error = 10 - 8 = 2
Squared error = 2² = 4
```


In plain English:

For every prediction, find the difference: (ŷ - y)

Square that difference to make it positive: (ŷ - y)²

Take the average of all those squared differences.

In [1]:
import torch

# -----------------------------
# 1. Create the same dataset
# -----------------------------

N = 10
D_in = 1
D_out = 1

# Input data
X = torch.randn(N, D_in)

# "True" relationship used to create our data:
# y = 2x + 1
true_W = torch.tensor([[2.0]])
true_b = torch.tensor(1.0)

# Actual target values
# Small random noise is added to make the data imperfect
y_true = X @ true_W + true_b + torch.randn(N, D_out) * 0.1


# -----------------------------
# 2. Create a model with random parameters
# -----------------------------

W = torch.randn(D_in, D_out, requires_grad=True)
b = torch.randn(1, requires_grad=True)


# -----------------------------
# 3. Forward pass
# -----------------------------

# Model prediction:
# y_hat = XW + b
y_hat = X @ W + b


# -----------------------------
# 4. Calculate MSE Loss
# -----------------------------

# Difference between prediction and actual value
error = y_hat - y_true

# Square the errors
squared_error = error ** 2

# Average all squared errors
loss = squared_error.mean()


# -----------------------------
# 5. Display the result
# -----------------------------

print("Prediction y_hat (first 3):")
print(y_hat[:3])

print("\nActual y_true (first 3):")
print(y_true[:3])

print("\nMSE Loss:")
print(loss)

Prediction y_hat (first 3):
tensor([[-0.1106],
        [ 1.3126],
        [-2.0789]], grad_fn=<SliceBackward0>)

Actual y_true (first 3):
tensor([[3.3608],
        [1.1897],
        [6.0435]])

MSE Loss:
tensor(24.6431, grad_fn=<MeanBackward0>)


## NOTICE THAT THIS LOSS TENSOR ALSO HAS A GRAD_FN.

It knows it's the result of all the previous calculations. It sits at the very end of our computation graph.

It is the source from which all knowledge will flow backward.


# NOW FOR THE MAGIC.

How do we know which way to turn the W and b knobs to lower that score?

## AUTOGRAD DOES ALL THE HEAVY LIFTING FOR US. WITH ONE SINGLE COMMAND.

WE ARE TELLING PYTORCH:

"Travel backward from loss and
calculate gradients for all parameters
with requires_grad=True."


## In Our Case

It will compute the two most important values we need:

$$
\frac{\partial L}{\partial W}
$$

The gradient of the Loss w.r.t. our Weight `W`.

$$
\frac{\partial L}{\partial b}
$$

The gradient of the Loss w.r.t. our Bias `b`.

In [2]:
# Compute gradients
loss.backward()

# The gradients are now stored in the .grad attribute which hold signal about how to adjust the knobs
print(f"Gradient for W (∂L/∂W):\n{W.grad}\n") # the directions
print(f"Gradient for b (∂L/∂b):\n{b.grad}") # the directions

Gradient for W (∂L/∂W):
tensor([[-14.8640]])

Gradient for b (∂L/∂b):
tensor([-3.1967])


# `w.grad = -1.0185`: The Sign Is Everything

- **Negative gradient** → increasing `w` **decreases** loss.
- Gradient points toward the steepest **increase**.
- Go in the **opposite direction** to minimize loss.

# `b.grad = -2.0673`: It's the Same Story

**Larger magnitude = steeper slope.**

# We Have What We Need to Improve

- **Measure error:** `loss`
- **Know direction:** `.grad`

# What Does the Gradient Tell Us?

- **Which value to adjust?** → The parameter whose `.grad` you are looking at (`w.grad`, `b.grad`, etc.).
- **How much to adjust?** → The **magnitude** of the gradient tells how steeply the loss changes.
- **Which direction?** → The **sign** tells the direction; to minimize loss, move **opposite to the gradient**.

> **Gradient = which parameter + how much + which direction**

# Gradient Descent — Mountain Analogy

Imagine we are **standing on a foggy mountain**:

- **Loss** → tells us our **current altitude** (where we are).
- **Gradient** → tells us the direction of the **steepest uphill slope**.
- We want to go **downhill**, so we move in the **opposite direction** of the gradient.
- We take a **small step**, measure the new loss, and repeat.

> **Loss = Where am I?**  
> **Gradient = Which way is uphill?**  
> **Gradient Descent = Take small steps downhill.**

# The Gradient Descent Update Formula

$$
\theta_{t+1} = \theta_t - \eta \cdot \nabla_\theta L
$$

- **θ (theta)** → represents all our parameters. For us, that's **`w` and `b`**.
- **η (eta)** → the **learning rate**. It controls how large a step we take (e.g., `0.01`).
- **∇θL** → the **gradient of the loss**. We calculate this using **`w.grad`** and **`b.grad`**.

> **Current parameter − (learning rate × gradient) = Updated parameter**

# The Update Rules in Code

```python
w_new = w_old - learning_rate * w.grad
b_new = b_old - learning_rate * b.grad
```

- **`w.grad`** → tells how to adjust `w`.
- **`b.grad`** → tells how to adjust `b`.
- **`learning_rate`** → controls how big the adjustment is.
- **`-`** → moves in the opposite direction of the gradient to reduce loss.

> **New value = Old value − (Learning rate × Gradient)**

# The Training Loop

Repeat the **5 training steps** for multiple epochs.

- **`torch.no_grad()`** → prevents PyTorch from tracking parameter updates and building a computation graph.
- **`.grad.zero_()`** → resets gradients before the next iteration so gradients don't accumulate.

> **Training loop = Forward → Loss → Gradients → Update → Repeat**

In [5]:
# Hyperparameters
learning_rate, epochs = 0.01, 100

# Re-initialize parameters
W, b = torch.randn(1, 1, requires_grad=True), torch.randn(1, requires_grad=True)

# Training Loop
for epoch in range(epochs):
    # Forward pass and loss | (y_hat -> guess)
    y_hat = X @ W + b
    loss = torch.mean((y_hat - y_true) ** 2) # calculate loss

    # Backward pass | look for blame for error gradient
    loss.backward()

    # Update parameters | gradient descent | nudging the parameters
    with torch.no_grad():
        W -= learning_rate * W.grad
        b -= learning_rate * b.grad

    # Zero gradients
    W.grad.zero_()
    b.grad.zero_()

# Resetting Gradients

`W.grad.zero_()`

`b.grad.zero_()`

These lines **reset the gradients to 0** after the current update.

## Why?

1. `loss.backward()` calculates the **new gradients**.

2. `W -= learning_rate * W.grad` updates `W`.

3. `W.grad.zero_()` clears the **old gradient**.

4. `b.grad.zero_()` clears the **old gradient** for `b`.

5. In the next iteration, `loss.backward()` calculates **fresh gradients** based on the **new `W` and `b`**.

## Why do we need `zero_()`?

PyTorch **accumulates gradients by default**.

Without `zero_()`:

**Iteration 1:**
`gradient = 2`

**Iteration 2:**
`gradient = 3`

PyTorch would accumulate them:

`stored gradient = 2 + 3 = 5`

But we want the gradient for the **current iteration only**.

With `zero_()`:

`Iteration 1 → calculate gradient → update → reset`

`Iteration 2 → calculate fresh gradient → update → reset`

> **Zero gradients → Calculate fresh gradients → Update parameters → Repeat**

# Why `torch.no_grad()`?

`torch.no_grad()` tells PyTorch **not to track the parameter update** in the computation graph.

    with torch.no_grad():
        W -= learning_rate * W.grad
        b -= learning_rate * b.grad

- `loss.backward()` → **calculates gradients**.
- `torch.no_grad()` → **updates `W` and `b` without tracking the update**.
- This prevents unnecessary computation-graph creation during parameter updates.

> **Calculate gradients → Update parameters without tracking → Repeat**

# Watching the Model Learn

Let's add a print statement inside the training loop to observe how the model parameters and loss change over time.

    if epoch % 10 == 0:
        print(f"Epoch {epoch:02d}: Loss={loss.item():.4f}, W={W.item():.3f}, b={b.item():.3f}")

After training, we can compare the learned parameters with the true parameters:

    print(f"\nFinal Parameters: W={W.item():.3f}, b={b.item():.3f}")
    print(f"True Parameters: W=2.000, b=1.000")

- `epoch % 10 == 0` → prints the values every **10 epochs**.
- `loss.item()` → shows the current **loss**.
- `W.item()` and `b.item()` → show the current **parameter values**.
- At the end, we compare the learned `W` and `b` with the **true values**.

> We can watch the **loss decrease** and `W` and `b` move toward their true values.

Since  training loop is learning a simple linear relationship:

y=2x+1

we can generate training data from that relationship, optionally adding a little noise.

In [9]:
import torch

# ---------------------------------------------------------
# Training data for a simple LINEAR REGRESSION problem
# ---------------------------------------------------------

# X = input / feature values
# Each row represents one training example.
X = torch.tensor([
    [1.0],
    [2.0],
    [3.0],
    [4.0],
    [5.0]
])

# y_true = target / actual output values
# These are the correct answers the model should learn to predict.
#
# We generate them using the true relationship:
# y = 2x + 1
y_true = 2 * X + 1


# ---------------------------------------------------------
# Display the training data
# ---------------------------------------------------------

print("X (input features):")
print(X)

print("\ny_true (target / correct outputs):")
print(y_true)

X (input features):
tensor([[1.],
        [2.],
        [3.],
        [4.],
        [5.]])

y_true (target / correct outputs):
tensor([[ 3.],
        [ 5.],
        [ 7.],
        [ 9.],
        [11.]])


In [10]:
import torch

# ============================================================
# SIMPLE LINEAR REGRESSION
#
# We have training data:
#
# X       = input feature
# y_true  = correct/target output
#
# Our data follows:
# y = 2x + 1
#
# The model does NOT know that W=2 and b=1.
# It starts with random W and b and learns them.
# ============================================================


# ------------------------------------------------------------
# 1. HYPERPARAMETERS
# ------------------------------------------------------------

learning_rate = 0.01   # How big a step to take during learning
epochs = 100            # How many times to go through the training data


# ------------------------------------------------------------
# 2. INITIAL MODEL PARAMETERS
# ------------------------------------------------------------

# Start with random values.
# The model will learn better values for W and b.

W = torch.randn(1, 1, requires_grad=True)   # Weight / slope
b = torch.randn(1, requires_grad=True)      # Bias / intercept


# ------------------------------------------------------------
# 3. TRAINING LOOP
# ------------------------------------------------------------

for epoch in range(epochs):

    # --------------------------------------------------------
    # STEP 1: FORWARD PASS
    # --------------------------------------------------------
    #
    # The model makes a prediction using:
    #
    #     y_hat = X * W + b
    #
    # y_hat = model's prediction / guess
    #
    # X      = input
    # W      = learned weight
    # b      = learned bias
    # --------------------------------------------------------

    y_hat = X @ W + b


    # --------------------------------------------------------
    # STEP 2: CALCULATE LOSS
    # --------------------------------------------------------
    #
    # Compare:
    #
    #     y_hat  = model's prediction
    #     y_true = correct answer
    #
    # MSE = Mean Squared Error
    #
    # Smaller loss = better predictions
    # --------------------------------------------------------

    loss = torch.mean((y_hat - y_true) ** 2)


    # --------------------------------------------------------
    # STEP 3: BACKWARD PASS
    # --------------------------------------------------------
    #
    # Calculate how much W and b contributed to the loss.
    #
    # PyTorch stores these gradients in:
    #
    #     W.grad
    #     b.grad
    #
    # The gradient tells:
    #   - which parameter needs adjustment
    #   - which direction to move
    #   - how steeply the loss changes
    # --------------------------------------------------------

    loss.backward()


    # --------------------------------------------------------
    # STEP 4: UPDATE PARAMETERS
    # --------------------------------------------------------
    #
    # Gradient Descent:
    #
    #     new value = old value - learning_rate * gradient
    #
    # We move in the opposite direction of the gradient
    # because we want to REDUCE the loss.
    # --------------------------------------------------------

    with torch.no_grad():

        W -= learning_rate * W.grad
        b -= learning_rate * b.grad


    # --------------------------------------------------------
    # STEP 5: RESET GRADIENTS
    # --------------------------------------------------------
    #
    # PyTorch accumulates gradients by default.
    #
    # We clear the old gradients so that the next
    # loss.backward() calculates fresh gradients.
    # --------------------------------------------------------

    W.grad.zero_()
    b.grad.zero_()


    # --------------------------------------------------------
    # WATCH THE MODEL LEARN
    # --------------------------------------------------------

    if epoch % 10 == 0:

        print(
            f"Epoch {epoch:02d} | "
            f"Loss = {loss.item():.4f} | "
            f"W = {W.item():.3f} | "
            f"b = {b.item():.3f}"
        )


# ------------------------------------------------------------
# AFTER TRAINING
# ------------------------------------------------------------

print("\nFinal learned parameters:")
print(f"W = {W.item():.3f}")
print(f"b = {b.item():.3f}")

print("\nTrue parameters:")
print("W = 2.000")
print("b = 1.000")

Epoch 00 | Loss = 52.8256 | W = 0.631 | b = -0.096
Epoch 10 | Loss = 0.3142 | W = 2.074 | b = 0.327
Epoch 20 | Loss = 0.0720 | W = 2.165 | b = 0.376
Epoch 30 | Loss = 0.0663 | W = 2.166 | b = 0.398
Epoch 40 | Loss = 0.0620 | W = 2.161 | b = 0.418
Epoch 50 | Loss = 0.0579 | W = 2.156 | b = 0.438
Epoch 60 | Loss = 0.0541 | W = 2.151 | b = 0.457
Epoch 70 | Loss = 0.0506 | W = 2.146 | b = 0.475
Epoch 80 | Loss = 0.0473 | W = 2.141 | b = 0.492
Epoch 90 | Loss = 0.0442 | W = 2.136 | b = 0.509

Final learned parameters:
W = 2.132
b = 0.524

True parameters:
W = 2.000
b = 1.000


# Watch the Learning Happen

- **Loss** → plummeting 📉
- **W** → approaching `2.0`
- **b** → approaching `1.0`

> The model is learning by reducing the loss and adjusting `W` and `b` toward the true values.

# What We Just Built

We just built a **Gradient Descent algorithm from scratch**.

We manually implemented:

- Parameter initialization (`W`, `b`)
- Forward pass
- Loss calculation
- Backward pass / gradient calculation
- Parameter updates using gradient descent
- Gradient resetting

> **We just built the core training loop of a machine learning model from scratch.**

WE HAD LOOSE TENSORS NAMED `W` AND `B`.

WE MANUALLY UPDATED THEM.

WE MANUALLY ZEROED THEIR GRADIENTS.

IMAGINE YOUR MODEL HAS 50 LAYERS. A MILLION PARAMETERS.

ARE YOU GOING TO WRITE A MILLION LINES OF CODE?